<a href="https://colab.research.google.com/github/PrathyushaS7/StockForecast_LSTM_GRU/blob/main/Stock_Forecasting_LSTM_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing necessary libraries
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.optimizers import Adam

# Configuring parameters and paths
DRIVE_MOUNT_POINT = '/content/drive'
BASE_PATH = '/content/drive/My Drive/StockForecast_LSTM_GRU/'
INPUT_CSV = BASE_PATH + 'data/Dataset.csv'

RESULTS_DATA_PATH = BASE_PATH + 'results/results_data/'
RESULTS_PLOTS_PATH = BASE_PATH + 'results/results_plots/'
RESULTS_SUMMARY_PATH = BASE_PATH + 'results/results_summary/'

LOOKBACK_WINDOW = 60
TRAIN_SPLIT_PERCENTAGE = 0.8
LSTM_UNITS_L1 = 100
LSTM_UNITS_L2 = 50
DENSE_UNITS = 25
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
EPOCHS = 150
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 10
RANDOM_SEED = 42

# Setting seeds for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Mounting Google Drive
try:
    drive.mount(DRIVE_MOUNT_POINT, force_remount=True)
except Exception as e:
    print(f"Drive mount failed: {e}")
    exit()

# Creating directories for saving results
os.makedirs(BASE_PATH + 'data/', exist_ok=True)
os.makedirs(RESULTS_DATA_PATH, exist_ok=True)
os.makedirs(RESULTS_PLOTS_PATH, exist_ok=True)
os.makedirs(RESULTS_SUMMARY_PATH, exist_ok=True)

In [ ]:
# Loading data
print(f"Loading data from: {INPUT_CSV}")
try:
    df_final = pd.read_csv(INPUT_CSV)
    df_final['Date'] = pd.to_datetime(df_final['Date'])
    df_final.sort_values(by=['Stock', 'Date'], inplace=True)
    df_final.reset_index(drop=True, inplace=True)
    print(f"Data shape: {df_final.shape}")
except FileNotFoundError:
    print(f"FileNotFoundError: {INPUT_CSV} missing.")
    exit()
except Exception as e:
    print(f"Data load failed: {e}")
    exit()

# Defining sequence generation function
def create_sequences(features: np.ndarray, target: np.ndarray, lookback: int):
    X, y = [], []
    if len(features) <= lookback:
        return np.array(X), np.array(y)
    for i in range(lookback, len(features)):
        X.append(features[i - lookback:i, :])
        y.append(target[i, 0])
    return np.array(X), np.array(y)

In [ ]:
# Defining LSTM model architecture
def build_lstm_model(input_shape):
    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_SEED)

    model = Sequential(name=f"LSTM_Model_{input_shape[1]}f")
    model.add(LSTM(LSTM_UNITS_L1, return_sequences=True, input_shape=input_shape, name="LSTM_1"))
    model.add(Dropout(DROPOUT_RATE, name="Dropout_1"))
    model.add(LSTM(LSTM_UNITS_L2, return_sequences=False, name="LSTM_2"))
    model.add(Dropout(DROPOUT_RATE, name="Dropout_2"))
    model.add(Dense(DENSE_UNITS, activation='relu', name="Dense_Intermediate"))
    model.add(Dense(1, name="Output"))
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=MeanSquaredError())
    return model

# Defining GRU model architecture
def build_gru_model(input_shape):
    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_SEED)

    model = Sequential(name=f"GRU_Model_{input_shape[1]}f")
    model.add(GRU(LSTM_UNITS_L1, return_sequences=True, input_shape=input_shape, name="GRU_1"))
    model.add(Dropout(DROPOUT_RATE, name="Dropout_1"))
    model.add(GRU(LSTM_UNITS_L2, return_sequences=False, name="GRU_2"))
    model.add(Dropout(DROPOUT_RATE, name="Dropout_2"))
    model.add(Dense(DENSE_UNITS, activation='relu', name="Dense_Intermediate"))
    model.add(Dense(1, name="Output"))
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=MeanSquaredError())
    return model

In [ ]:
# Initializing processing loop
all_results = []
unique_stocks = df_final['Stock'].unique()
print(f"\nFound {len(unique_stocks)} stocks: {', '.join(unique_stocks)}")

# Processing each stock
for stock in unique_stocks:
    print(f"\nProcessing stock: {stock}")
    df_stock = df_final[df_final['Stock'] == stock].copy()
    df_stock.reset_index(drop=True, inplace=True)

    min_data_len = LOOKBACK_WINDOW + 20
    if len(df_stock) < min_data_len:
        print(f"Skipping {stock}: Insufficient data ({len(df_stock)} rows)")
        continue

    variants = {}

    # Preparing no sentiment variant
    features_no_sent = ['Close', 'Volume', 'RSI', 'MACD']
    df_no_sent = df_stock[['Date'] + features_no_sent].dropna().reset_index(drop=True)
    if len(df_no_sent) >= min_data_len:
        variants['no_sentiment'] = {'df': df_no_sent, 'features': features_no_sent}
    else:
        print(f"Skipping {stock}: Sparse data post-NaN drop (no_sentiment)")

    # Preparing forward fill variant
    features_fill = features_no_sent + ['General Sentiment Score']
    df_ff = df_stock[['Date'] + features_fill].copy()
    df_ff.dropna(subset=features_no_sent, inplace=True)
    df_ff['General Sentiment Score'] = df_ff['General Sentiment Score'].bfill().ffill().fillna(0)
    df_ff.reset_index(drop=True, inplace=True)

    if len(df_ff) >= min_data_len:
        variants['forward_fill'] = {'df': df_ff, 'features': features_fill}
    else:
        print(f"Skipping {stock}: Sparse data post-cleaning (forward_fill)")

    # Training models per variant
    for variant_name, variant in variants.items():
        df_variant = variant['df']
        features_list = variant['features']
        target_col = 'Close'

        data_len = len(df_variant)
        split_idx = int(data_len * TRAIN_SPLIT_PERCENTAGE)
        if split_idx >= data_len - LOOKBACK_WINDOW:
            split_idx = data_len - LOOKBACK_WINDOW - 1
            if split_idx < LOOKBACK_WINDOW:
                print(f"Skipping {stock} {variant_name}: Improper split index")
                continue

        df_train = df_variant.iloc[:split_idx]
        df_test = df_variant.iloc[split_idx:]

        # Scaling data
        scaler = MinMaxScaler((0, 1)).fit(df_train[features_list])
        scaled_train = scaler.transform(df_train[features_list])
        scaled_test = scaler.transform(df_test[features_list])

        tgt_idx = features_list.index(target_col)
        scaled_train_tgt = scaled_train[:, tgt_idx:tgt_idx + 1]
        scaled_test_tgt = scaled_test[:, tgt_idx:tgt_idx + 1]

        X_train, y_train = create_sequences(scaled_train, scaled_train_tgt, LOOKBACK_WINDOW)
        X_test, y_test = create_sequences(scaled_test, scaled_test_tgt, LOOKBACK_WINDOW)

        if X_train.size == 0 or X_test.size == 0:
            print(f"Skipping {stock} {variant_name}: Insufficient sequence data")
            continue

        n_features = X_train.shape[2]

        for model_type in ['LSTM', 'GRU']:
            build_fn = build_lstm_model if model_type == 'LSTM' else build_gru_model
            model = build_fn((LOOKBACK_WINDOW, n_features))

            es = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)
            print(f"Training {model_type} for {stock} ({variant_name})...")
            history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                                validation_data=(X_test, y_test), callbacks=[es], verbose=0)

            # Saving training history
            history_df = pd.DataFrame(history.history)
            history_filename = f"{RESULTS_DATA_PATH}{stock}_{variant_name}_{model_type}_training_history.csv"
            history_df.to_csv(history_filename, index_label='Epoch')

            # Plotting and saving loss curves
            plt.style.use('seaborn-v0_8-darkgrid')
            plt.figure(figsize=(12, 5))
            plt.plot(history.history['loss'], label='Training')
            plt.plot(history.history['val_loss'], label='Validation')
            plt.title(f'Loss ({stock} - {variant_name} - {model_type})')
            plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend();
            plot_filename_loss = f"{RESULTS_PLOTS_PATH}{stock}_{variant_name}_{model_type}_loss_curve.png"
            plt.savefig(plot_filename_loss)
            plt.close()

            # Generating predictions
            y_pred_scaled = model.predict(X_test, verbose=0)
            if y_pred_scaled.ndim == 1:
                y_pred_scaled = y_pred_scaled.reshape(-1, 1)

            dummy_pred = np.zeros((len(y_pred_scaled), scaler.n_features_in_))
            dummy_pred[:, tgt_idx] = y_pred_scaled.flatten()
            y_pred_inv = scaler.inverse_transform(dummy_pred)[:, tgt_idx]

            if y_test.ndim == 1:
                y_test = y_test.reshape(-1, 1)
            dummy_true = np.zeros((len(y_test), scaler.n_features_in_))
            dummy_true[:, tgt_idx] = y_test.flatten()
            y_true_inv = scaler.inverse_transform(dummy_true)[:, tgt_idx]

            # Calculating metrics
            rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
            mae = mean_absolute_error(y_true_inv, y_pred_inv)

            orig_close = df_test[target_col].values
            if len(orig_close) > LOOKBACK_WINDOW:
                prev = orig_close[LOOKBACK_WINDOW - 1:-1]
                if len(prev) == len(y_pred_inv):
                    actual_chg = np.sign(y_true_inv - prev)
                    pred_chg = np.sign(y_pred_inv - prev)
                    dir_acc = np.mean(actual_chg == pred_chg) * 100
                else:
                    dir_acc = np.nan
            else:
                dir_acc = np.nan

            # Appending results
            all_results.append({
                'Model': model_type,
                'Stock': stock,
                'Variant': variant_name,
                'RMSE': rmse,
                'MAE': mae,
                'Directional_Accuracy': dir_acc,
                'Epochs_Trained': len(history.history['loss'])
            })

            if len(df_test) >= LOOKBACK_WINDOW and len(y_true_inv) == len(y_pred_inv):
                dates = df_test['Date'].iloc[LOOKBACK_WINDOW:LOOKBACK_WINDOW + len(y_true_inv)]

                # Saving actual vs predicted data
                predictions_data = pd.DataFrame({
                    'Date': dates,
                    'Actual': y_true_inv,
                    'Predicted': y_pred_inv
                })
                predictions_filename = f"{RESULTS_DATA_PATH}{stock}_{variant_name}_{model_type}_actual_vs_predicted.csv"
                predictions_data.to_csv(predictions_filename, index=False)

# Formatting overall summary output
print("\nOverall Results Summary:")

if not all_results:
    print("No results generated.")
else:
    results_df = pd.DataFrame(all_results)
    results_df = results_df.round({'RMSE': 4, 'MAE': 4, 'Directional_Accuracy': 2})
    results_df.sort_values(by=['Stock', 'Variant', 'Model'], inplace=True)

    # Saving overall results summary
    overall_results_filename = f"{RESULTS_SUMMARY_PATH}overall_model_performance_summary.csv"
    results_df.to_csv(overall_results_filename, index=False)

    summary = (results_df.groupby(['Model', 'Variant'])[['RMSE', 'MAE', 'Directional_Accuracy']]
               .mean().round({'RMSE': 4, 'MAE': 4, 'Directional_Accuracy': 2}))

    # Saving grouped summary
    summary_filename_grouped = f"{RESULTS_SUMMARY_PATH}average_errors_summary_grouped.csv"
    summary.to_csv(summary_filename_grouped)

    print("\nAverage Errors Across All Stocks:")
    print(summary.to_string())

In [ ]:
# Generating combined plots
print("\nGenerating combined plots...")

# Setting plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 22
})

colors = ['#0072B2', '#009E73', '#56B4E9', '#E69F00', '#D55E00']

for stock in unique_stocks:
    file_paths = {
        "lstm_ns": f"{RESULTS_DATA_PATH}{stock}_no_sentiment_LSTM_actual_vs_predicted.csv",
        "lstm_s":  f"{RESULTS_DATA_PATH}{stock}_forward_fill_LSTM_actual_vs_predicted.csv",
        "gru_ns":  f"{RESULTS_DATA_PATH}{stock}_no_sentiment_GRU_actual_vs_predicted.csv",
        "gru_s":   f"{RESULTS_DATA_PATH}{stock}_forward_fill_GRU_actual_vs_predicted.csv",
    }

    data_frames = {}

    # Loading base actuals from first available model
    try:
        df = pd.read_csv(file_paths["lstm_ns"])
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        data_frames['Actual'] = df['Actual']
        data_frames['LSTM No Sentiment'] = df['Predicted']
    except FileNotFoundError:
        print(f"Base file missing for {stock} ({file_paths['lstm_ns']}). Skipping plot.")
        continue

    # Loading variant predictions
    model_variants_to_load = {
        "LSTM Sentiment": file_paths["lstm_s"],
        "GRU No Sentiment": file_paths["gru_ns"],
        "GRU Sentiment": file_paths["gru_s"],
    }

    for label, path in model_variants_to_load.items():
        try:
            df_model = pd.read_csv(path)
            df_model['Date'] = pd.to_datetime(df_model['Date'])
            df_model.set_index('Date', inplace=True)
            data_frames[label] = df_model['Predicted']
        except FileNotFoundError:
            pass # Silently skip missing variants

    plot_df = pd.DataFrame(data_frames)

    # Creating plot architecture
    fig, ax = plt.subplots(figsize=(18, 9))

    if 'Actual' in plot_df.columns:
        ax.plot(plot_df.index, plot_df['Actual'], label='Actual Price', color=colors[0], linewidth=2.5, linestyle='-')

    model_plot_labels = {
        'LSTM No Sentiment': colors[1],
        'LSTM Sentiment': colors[2],
        'GRU No Sentiment': colors[3],
        'GRU Sentiment': colors[4],
    }

    for label, color in model_plot_labels.items():
        if label in plot_df.columns:
            ax.plot(plot_df.index, plot_df[label], label=label, color=color, linewidth=1.8, linestyle='--')

    ax.set_title(f'{stock} - Actual vs. Predicted Prices')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45, ha='right')
    ax.legend(loc='upper left')
    plt.tight_layout()

    # Saving final output
    plot_save_path = f"{RESULTS_PLOTS_PATH}{stock}_combined_predictions.png"
    try:
        plt.savefig(plot_save_path, dpi=150)
    except Exception as e:
        print(f"Plot save failed for {stock}: {e}")
    plt.close()

print(f"\nFinish. Outputs saved in {BASE_PATH}results/")